# Multi-Dataset Comparison: CNN vs. fine-tuned ResNet vs. Phikon-v2 linear probe

**Goal:** test whether the pattern from the TCGA-LUAD TIL experiment (a frozen pathology foundation-model embedding + linear probe beats a from-scratch CNN and a fine-tuned ImageNet ResNet on accuracy, with AUROC roughly tied) holds on other datasets, including multi-class ones.

**How this notebook is organised**
1. **Config** – budgets, seeds, paths, per-run overrides.
2. **Data contract** – every dataset is reduced to one `LabeledPatches` object (images, integer labels, a group id per image, a train/test flag). Nothing below this point knows which dataset it is looking at.
3. **Models** – the same three models as the original notebooks, generalised from 1 logit to `n_classes` logits.
4. **Evaluation** – accuracy vs. majority baseline, balanced accuracy, macro-F1, one-vs-rest AUROC, per-class recall/F1, confusion matrices, and a *group-level* bootstrap with paired differences.
5. **Adapters** – one small function per dataset that builds a `LabeledPatches`. Adapters so far: `synthetic` (a smoke test for this code path, **not** a result) and `nct_crc_he`.
6. **Run + summary** – runs the chosen datasets and writes one JSON per dataset plus a combined table.

**Things to keep in mind when comparing to the earlier notebooks**
- The CNN and ResNet here are trained on the *same subsample as the probe* (`n_train`), not on all 80k TCGA patches the original notebook used. That is deliberate (equal budget across models and datasets), but it means numbers are **not directly comparable** to the original run; re-run TCGA-LUAD through this pipeline for an apples-to-apples row.
- Where a dataset has no patient ID, the group is a slide/image id or, failing that, the patch itself (`group_kind='patch'`). Bootstrap CIs then resample patches, which is **too optimistic** and is labelled as such in the results.
- Phikon-v2 was pretrained on public TCGA slides, so TCGA-derived datasets can favour the probe. Each adapter records a contamination note that is carried into the results.

In [ ]:
# transformers is only needed for the Phikon-v2 embeddings (Colab already ships torch/sklearn)
!pip install -q transformers

In [ ]:
import os, sys, json, time, hashlib, warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.models import resnet18, ResNet18_Weights
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, StratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')   # LogisticRegressionCV deprecation chatter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Everything the notebook writes (checkpoints, cached embeddings, results) lives under ROOT.
ROOT = os.environ.get('MD_ROOT') or (
    '/content/drive/MyDrive/Multi_Dataset_TIL' if IN_COLAB else os.path.abspath('./multi_dataset_outputs'))
for sub in ('checkpoints', 'embeddings', 'results'):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("ROOT:", ROOT, "| Device:", DEVICE)

## 1. Config

`REAL_CFG` mirrors the settings of the original TCGA-LUAD notebooks (CNN: Adam 1e-3, 10 epochs, 64 px; ResNet-18: Adam 1e-4, 5 epochs, 224 px; probe: 20k train / 5k test embeddings). `SMOKE_CFG` is a tiny, offline-friendly version used only for the `synthetic` smoke test (random-init ResNet, stub embedder, small image sizes).

In [ ]:
SEED = 0
STORE_SIZE = 96          # adapters store patches at this size (HxW) so memory stays bounded across datasets

REAL_CFG = dict(
    n_train=20_000, n_test=5_000,             # same subsample for ALL three models
    cnn_size=64, cnn_epochs=10, cnn_lr=1e-3, cnn_batch=128,
    resnet_size=224, resnet_epochs=5, resnet_lr=1e-4, resnet_batch=64, resnet_pretrained=True,
    embedder='phikon', probe_Cs=[1e-4, 1e-3, 1e-2, 1e-1, 1], cv_folds=5, probe_fallback_C=1e-3,
    n_boot=2000, num_workers=2,
)
SMOKE_CFG = dict(
    REAL_CFG,
    n_train=1_500, n_test=600,
    cnn_epochs=4, resnet_size=64, resnet_epochs=1, resnet_pretrained=False,
    embedder='stub', n_boot=300, num_workers=0,
)

def cfg_for(name):
    return SMOKE_CFG if name == 'synthetic' else REAL_CFG

MODELS = ['CNN', 'ResNet', 'Probe']
METRICS = ['accuracy', 'balanced_acc', 'macro_f1', 'auroc']
PAIRS = [('Probe', 'CNN'), ('Probe', 'ResNet'), ('ResNet', 'CNN')]   # (a, b) -> a minus b

## 2. Data contract and split helpers

An adapter must return a `LabeledPatches`:

| field | meaning |
|---|---|
| `X` | `(N, S, S, 3)` uint8 RGB patches |
| `y` | `(N,)` integer labels in `[0, n_classes)` |
| `groups` | `(N,)` string id of the patient (best), slide, or image each patch came from |
| `group_kind` | `'patient'`, `'slide'`, `'image'`, or `'patch'` (= no grouping available) |
| `is_test` | `(N,)` bool; the official split if one exists, else from `grouped_split` |
| `note` | free-text contamination / caveat note carried into the results |

`run_experiment` refuses to run if any non-`'patch'` group appears in both train and test.

In [ ]:
@dataclass
class LabeledPatches:
    name: str
    X: np.ndarray
    y: np.ndarray
    class_names: list
    groups: np.ndarray
    group_kind: str
    is_test: np.ndarray
    note: str = ''

    def __post_init__(self):
        n = len(self.y)
        assert self.X.ndim == 4 and self.X.shape[0] == n and self.X.shape[-1] == 3, f"bad X shape {self.X.shape}"
        assert self.X.dtype == np.uint8, "X must be uint8"
        assert len(self.groups) == n and len(self.is_test) == n
        assert self.group_kind in ('patient', 'slide', 'image', 'patch')
        self.y = np.asarray(self.y).astype(int)
        self.groups = np.asarray(self.groups).astype(str)
        self.is_test = np.asarray(self.is_test).astype(bool)
        assert self.y.min() >= 0 and self.y.max() < len(self.class_names), "labels outside class_names range"

    @property
    def n_classes(self):
        return len(self.class_names)


def grouped_split(groups, test_frac=0.2, seed=SEED):
    """Boolean is_test mask where whole groups go to test until ~test_frac of patches is reached."""
    groups = np.asarray(groups).astype(str)
    rng = np.random.default_rng(seed)
    unique = np.array(sorted(set(groups)))
    rng.shuffle(unique)
    sizes = pd.Series(groups).value_counts()
    target, taken, test_groups = test_frac * len(groups), 0, set()
    for g in unique:
        if taken >= target:
            break
        test_groups.add(g)
        taken += sizes[g]
    return np.array([g in test_groups for g in groups])


def stratified_subsample(idx, y, n, rng):
    """Up to n indices from idx, keeping each class's natural share (so base rates stay realistic)."""
    idx = np.asarray(idx)
    if len(idx) <= n:
        return np.sort(idx)
    out = []
    classes, counts = np.unique(y[idx], return_counts=True)
    for c, k in zip(classes, counts):
        pool = idx[y[idx] == c]
        take = min(len(pool), max(1, int(round(n * k / len(idx)))))
        out.append(rng.choice(pool, take, replace=False))
    return np.sort(np.concatenate(out))


def describe(ds, tr=None, te=None):
    """Print class balance and group counts per split (used before every run)."""
    print(f"Dataset {ds.name}: {len(ds.y)} patches, {ds.n_classes} classes, groups={ds.group_kind} "
          f"({len(set(ds.groups))} unique)")
    for label, idx in [('train', np.where(~ds.is_test)[0] if tr is None else tr),
                       ('test', np.where(ds.is_test)[0] if te is None else te)]:
        counts = np.bincount(ds.y[idx], minlength=ds.n_classes)
        frac = ", ".join(f"{c}={k} ({k / max(1, counts.sum()):.0%})" for c, k in zip(ds.class_names, counts))
        print(f"  {label:5s} n={len(idx):6d} groups={len(set(ds.groups[idx])):4d} | {frac}")
    if ds.note:
        print("  note:", ds.note)

## 3. Models

Same three models as the original notebooks, with the head generalised from 1 logit (BCE + `pos_weight`) to `n_classes` logits (cross-entropy + balanced class weights, computed on the train subsample only).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class TorchPatches(Dataset):
    """uint8 patches -> float tensors, optionally resized and ImageNet-normalised."""
    def __init__(self, X, y, size, imagenet=False):
        self.X, self.y, self.size, self.imagenet = X, y, size, imagenet
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        img = torch.from_numpy(self.X[i]).permute(2, 0, 1).float() / 255.0
        if img.shape[-1] != self.size:
            img = TF.resize(img, [self.size, self.size], antialias=True)
        if self.imagenet:
            img = TF.normalize(img, IMAGENET_MEAN, IMAGENET_STD)
        return img, int(self.y[i])


class TinyCNN(nn.Module):
    """Same architecture as the original notebook (3 conv layers), with n_classes outputs."""
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


def make_resnet(n_classes, pretrained):
    net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    net.fc = nn.Linear(net.fc.in_features, n_classes)
    return net


def balanced_class_weights(y, n_classes):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    return torch.tensor(len(y) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32)


def fit_net(net, X, y, *, size, imagenet, epochs, lr, batch, n_classes, num_workers, ckpt, label):
    """Train (or load from ckpt). Checkpoint is written only after real training finishes."""
    net = net.to(DEVICE)
    if os.path.exists(ckpt):
        print(f"[{label}] found checkpoint, loading: {ckpt}")
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        return net, []
    loader = DataLoader(TorchPatches(X, y, size, imagenet), batch_size=batch, shuffle=True,
                        num_workers=num_workers)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=balanced_class_weights(y, n_classes).to(DEVICE))
    losses = []
    for epoch in range(epochs):
        net.train()
        total = 0.0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = crit(net(imgs), labels)
            loss.backward()
            opt.step()
            total += loss.item() * imgs.size(0)
        losses.append(total / len(y))
        print(f"[{label}] epoch {epoch + 1}/{epochs}: train loss = {losses[-1]:.4f}")
    torch.save(net.state_dict(), ckpt)
    return net, losses


@torch.no_grad()
def predict_proba(net, X, size, imagenet, batch=128, num_workers=0):
    net.eval()
    loader = DataLoader(TorchPatches(X, np.zeros(len(X), dtype=int), size, imagenet),
                        batch_size=batch, shuffle=False, num_workers=num_workers)
    out = [torch.softmax(net(imgs.to(DEVICE)), dim=1).cpu() for imgs, _ in loader]
    return torch.cat(out).numpy().astype(np.float64)

In [ ]:
# ---- Phikon-v2 embeddings (frozen backbone) + linear probe ----

def phikon_embedder():
    """CLS-token embeddings from owkin/phikon-v2 (ViT-L, DINOv2). Backbone is fully frozen."""
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
    model = AutoModel.from_pretrained("owkin/phikon-v2").to(DEVICE).eval()
    def embed(X, batch_size=128):
        out = []
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
            for i in range(0, len(X), batch_size):
                inputs = processor(images=list(X[i:i + batch_size]), return_tensors="pt").to(DEVICE)
                out.append(model(**inputs).last_hidden_state[:, 0, :].float().cpu().numpy())
                if (i // batch_size) % 20 == 0:
                    print(f"  embeddings {i + len(X[i:i + batch_size])}/{len(X)}")
        return np.concatenate(out)
    return embed


def stub_embedder(dim=64):
    """Offline stand-in used ONLY by the synthetic smoke test: random projection of a 8x8 thumbnail."""
    def embed(X, batch_size=None):
        n, h = len(X), X.shape[1]
        k = h // 8
        thumb = X[:, :k * 8, :k * 8].reshape(n, 8, k, 8, k, 3).mean(axis=(2, 4)).reshape(n, -1) / 255.0
        proj = np.random.default_rng(123).normal(size=(thumb.shape[1], dim))
        return thumb @ proj
    return embed


def get_embeddings(ds, tr, te, cfg, tag):
    path = os.path.join(ROOT, 'embeddings', f"{ds.name}_{cfg['embedder']}_{tag}.npz")
    if os.path.exists(path):
        print(f"[Probe] found cached embeddings: {path}")
        d = np.load(path)
        return d['train'], d['test']
    embed = phikon_embedder() if cfg['embedder'] == 'phikon' else stub_embedder()
    print("[Probe] extracting train embeddings..."); Etr = embed(ds.X[tr])
    print("[Probe] extracting test embeddings...");  Ete = embed(ds.X[te])
    np.savez_compressed(path, train=Etr, test=Ete)
    return Etr, Ete


def fit_probe(Etr, ytr, gtr, group_kind, n_classes, cfg):
    """Standardise (fit on train only) + class-weighted logistic regression; C chosen by CV.
    CV is grouped when real groups exist, so patches from one patient never sit in both folds."""
    scaler = StandardScaler().fit(Etr)
    Xtr = scaler.transform(Etr)
    n_groups = len(set(gtr))
    if group_kind == 'patch' or n_groups < 3:
        folds = list(StratifiedKFold(cfg['cv_folds'], shuffle=True, random_state=SEED).split(Xtr, ytr))
        cv_kind = 'stratified (NOT grouped: no group ids)'
    else:
        folds = list(GroupKFold(min(cfg['cv_folds'], n_groups)).split(Xtr, ytr, groups=gtr))
        cv_kind = f'grouped by {group_kind}'
    scoring = 'roc_auc' if n_classes == 2 else 'roc_auc_ovr'
    try:
        probe = LogisticRegressionCV(Cs=cfg['probe_Cs'], cv=folds, class_weight='balanced',
                                     max_iter=2000, scoring=scoring, n_jobs=-1).fit(Xtr, ytr)
        chosen_C = float(probe.C_[0])
        cv_scores = np.mean([v.mean(axis=0) for v in probe.scores_.values()], axis=0)
        print(f"[Probe] chosen C={chosen_C} ({cv_kind}); mean CV {scoring} per C:",
              dict(zip(cfg['probe_Cs'], cv_scores.round(4))))
    except ValueError as e:   # e.g. a fold is missing a rare class
        warnings.warn(f"CV failed ({e}); falling back to fixed C={cfg['probe_fallback_C']}")
        from sklearn.linear_model import LogisticRegression
        probe = LogisticRegression(C=cfg['probe_fallback_C'], class_weight='balanced', max_iter=2000).fit(Xtr, ytr)
        chosen_C, cv_kind = cfg['probe_fallback_C'], cv_kind + ' (CV failed, fixed C)'
    return scaler, probe, chosen_C, cv_kind


def probe_proba(scaler, probe, Ete, n_classes):
    P = np.zeros((len(Ete), n_classes))
    P[:, probe.classes_] = probe.predict_proba(scaler.transform(Ete))   # zero-fill classes absent from train
    return P

## 4. Evaluation

All three models are scored on the **same** test subsample. Metrics are computed from a confusion matrix / ranks (no sklearn in the loop) so the 2,000-resample bootstrap stays fast. Class-averaged metrics (`balanced_acc`, `macro_f1`, `auroc`) average over the classes **present in the evaluated sample**; AUROC is one-vs-rest (identical to the ordinary AUROC for 2 classes).

In [ ]:
def auroc_ovr(y, P):
    """Macro one-vs-rest AUROC over classes present in y, via rank statistics."""
    aucs = []
    for c in np.unique(y):
        pos = y == c
        n_pos, n_neg = pos.sum(), len(y) - pos.sum()
        if n_neg == 0:
            return np.nan
        r = rankdata(P[:, c])
        aucs.append((r[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
    return float(np.mean(aucs))


def confusion(y, P):
    C = P.shape[1]
    return np.bincount(y * C + P.argmax(1), minlength=C * C).reshape(C, C)


def metrics_from(y, P):
    cm = confusion(y, P)
    support, tp = cm.sum(1), np.diag(cm)
    present = support > 0
    recall = np.divide(tp, support, out=np.zeros(len(tp)), where=present)
    denom = cm.sum(1) + cm.sum(0)
    f1 = np.divide(2 * tp, denom, out=np.zeros(len(tp)), where=denom > 0)
    return dict(accuracy=float(tp.sum() / len(y)), balanced_acc=float(recall[present].mean()),
                macro_f1=float(f1[present].mean()), auroc=auroc_ovr(y, P)), recall, f1


def group_bootstrap(y, probs, groups, group_kind, n_boot, seed=SEED):
    """Resample whole groups (patients/slides/images) with replacement; every model sees the same resamples."""
    if len(set(groups)) < 5 and group_kind != 'patch':
        warnings.warn("fewer than 5 groups in test: falling back to patch-level bootstrap (too optimistic)")
        groups, group_kind = np.arange(len(y)).astype(str), 'patch'
    _, inv = np.unique(groups, return_inverse=True)
    idx_by_group = [np.where(inv == k)[0] for k in range(inv.max() + 1)]
    rng = np.random.default_rng(seed)
    boot = {m: {k: [] for k in METRICS} for m in probs}
    for _ in range(n_boot):
        idx = np.concatenate([idx_by_group[g] for g in rng.integers(0, len(idx_by_group), len(idx_by_group))])
        if len(np.unique(y[idx])) < 2:
            continue
        for m, P in probs.items():
            mt, _, _ = metrics_from(y[idx], P[idx])
            for k in METRICS:
                boot[m][k].append(mt[k])
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in boot.items()}, group_kind


def ci(a):
    return [float(x) for x in np.nanpercentile(a, [2.5, 97.5])]

## 5. Running one dataset

`run_experiment` takes a `LabeledPatches` and a config, trains/loads the three models on one shared train subsample, scores them on one shared test subsample, bootstraps over groups, prints a report, and writes `results/results_<name>.json`. Checkpoints and embeddings are cached under `ROOT` and keyed by dataset, budget, seed and the exact sample drawn, so re-running after a Colab disconnect resumes instead of retraining.

In [ ]:
def jsonable(o):
    if isinstance(o, dict):  return {k: jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):  return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):  return o.tolist()
    if isinstance(o, (np.floating, np.integer)):  return o.item()
    return o


def run_experiment(ds, cfg, seed=SEED):
    t0 = time.time()
    torch.manual_seed(seed); np.random.seed(seed)
    rng = np.random.default_rng(seed)
    C = ds.n_classes

    tr = stratified_subsample(np.where(~ds.is_test)[0], ds.y, cfg['n_train'], rng)
    te = stratified_subsample(np.where(ds.is_test)[0], ds.y, cfg['n_test'], rng)
    if ds.group_kind != 'patch':
        overlap = set(ds.groups[tr]) & set(ds.groups[te])
        assert not overlap, f"{len(overlap)} {ds.group_kind}(s) appear in both train and test, e.g. {[str(g) for g in sorted(overlap)[:3]]}"
    describe(ds, tr, te)
    assert len(np.unique(ds.y[te])) >= 2, "test subsample has fewer than 2 classes"
    Xtr, ytr, Xte, yte = ds.X[tr], ds.y[tr], ds.X[te], ds.y[te]

    tag = f"n{len(tr)}-{len(te)}_s{seed}_{hashlib.md5(tr.tobytes() + te.tobytes()).hexdigest()[:8]}"
    ck = lambda m, extra='': os.path.join(ROOT, 'checkpoints', f"{ds.name}_{m}_{tag}{extra}.pt")
    nw = cfg['num_workers']
    probs, train_losses = {}, {}

    # --- CNN (from scratch) ---
    torch.manual_seed(seed)
    net, train_losses['CNN'] = fit_net(TinyCNN(C), Xtr, ytr, size=cfg['cnn_size'], imagenet=False,
        epochs=cfg['cnn_epochs'], lr=cfg['cnn_lr'], batch=cfg['cnn_batch'], n_classes=C, num_workers=nw,
        ckpt=ck('cnn', f"_e{cfg['cnn_epochs']}_z{cfg['cnn_size']}"), label='CNN')
    probs['CNN'] = predict_proba(net, Xte, cfg['cnn_size'], False, num_workers=nw)

    # --- ResNet-18 (ImageNet-pretrained, fully fine-tuned) ---
    torch.manual_seed(seed)
    net, train_losses['ResNet'] = fit_net(make_resnet(C, cfg['resnet_pretrained']), Xtr, ytr,
        size=cfg['resnet_size'], imagenet=True, epochs=cfg['resnet_epochs'], lr=cfg['resnet_lr'],
        batch=cfg['resnet_batch'], n_classes=C, num_workers=nw,
        ckpt=ck('resnet', f"_e{cfg['resnet_epochs']}_z{cfg['resnet_size']}_p{int(cfg['resnet_pretrained'])}"),
        label='ResNet')
    probs['ResNet'] = predict_proba(net, Xte, cfg['resnet_size'], True, batch=64, num_workers=nw)
    del net
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # --- Frozen foundation-model embeddings + linear probe ---
    Etr, Ete = get_embeddings(ds, tr, te, cfg, tag)
    scaler, probe, chosen_C, cv_kind = fit_probe(Etr, ytr, ds.groups[tr], ds.group_kind, C, cfg)
    probs['Probe'] = probe_proba(scaler, probe, Ete, C)

    # --- Metrics, bootstrap, report ---
    baseline = float(np.bincount(yte, minlength=C).max() / len(yte))
    point, per_class, confs = {}, {}, {}
    for m in MODELS:
        point[m], rec, f1 = metrics_from(yte, probs[m])
        per_class[m] = {n: dict(recall=float(r), f1=float(f), support=int((yte == i).sum()))
                        for i, (n, r, f) in enumerate(zip(ds.class_names, rec, f1))}
        confs[m] = confusion(yte, probs[m])
    boot, boot_kind = group_bootstrap(yte, probs, ds.groups[te], ds.group_kind, cfg['n_boot'], seed)
    n_valid = len(boot['CNN']['accuracy'])

    res = dict(dataset=ds.name, class_names=ds.class_names, n_classes=C, group_kind=ds.group_kind,
               bootstrap_kind=boot_kind, n_test_groups=len(set(ds.groups[te])), n_boot_valid=n_valid,
               n_train=len(tr), n_test=len(te), majority_baseline=baseline, probe_C=chosen_C, probe_cv=cv_kind,
               note=ds.note, cfg=cfg, seed=seed, train_losses=train_losses, confusion=confs,
               per_class=per_class, metrics={}, paired={})
    for m in MODELS:
        res['metrics'][m] = {k: dict(point=point[m][k], lo=ci(boot[m][k])[0], hi=ci(boot[m][k])[1]) for k in METRICS}
    for a, b in PAIRS:
        res['paired'][f"{a}-{b}"] = {}
        for k in METRICS:
            d = boot[a][k] - boot[b][k]
            lo, hi = ci(d)
            res['paired'][f"{a}-{b}"][k] = dict(mean=float(np.nanmean(d)), lo=lo, hi=hi, excludes_zero=bool(lo > 0 or hi < 0))
    res['elapsed_s'] = time.time() - t0

    with open(os.path.join(ROOT, 'results', f"results_{ds.name}.json"), 'w') as f:
        json.dump(jsonable(res), f, indent=2)
    report(res)
    return res


def report(res):
    print(f"\n=== {res['dataset']}: {res['n_classes']} classes, majority baseline {res['majority_baseline']:.4f} ===")
    print(f"bootstrap over {res['n_test_groups']} test {res['group_kind']} group(s) [{res['bootstrap_kind']}], "
          f"{res['n_boot_valid']} valid resamples | probe C={res['probe_C']} ({res['probe_cv']})")
    rows = []
    for m in MODELS:
        row = {'model': m}
        for k in METRICS:
            v = res['metrics'][m][k]
            row[k] = f"{v['point']:.3f} [{v['lo']:.3f}, {v['hi']:.3f}]"
        rows.append(row)
    print(pd.DataFrame(rows).set_index('model').to_string())
    print("\nPaired differences (a - b), same resamples:")
    for pair, d in res['paired'].items():
        print(f"  {pair:14s} " + "  ".join(
            f"{k}: {d[k]['mean']:+.3f} [{d[k]['lo']:+.3f}, {d[k]['hi']:+.3f}]{'*' if d[k]['excludes_zero'] else ' '}"
            for k in METRICS))
    print("  (* = 95% CI excludes 0)")
    if res['group_kind'] == 'patch':
        print("  WARNING: no group ids, so these CIs resample patches and are too optimistic.")
    if res['note']:
        print("  note:", res['note'])

## 6. Adapters

An adapter is a function `build_<name>() -> LabeledPatches`. Register it in `ADAPTERS`. `synthetic` is a smoke test: 4 classes defined by a colour/texture signature, 24 "patients" that each add their own stain-like colour shift, so a patient-level split is genuinely harder than a patch-level one. It is only there to exercise the pipeline end to end offline; **its numbers mean nothing about pathology.**

In [ ]:
def build_synthetic(n_per_group=120, n_groups=24, n_classes=4, size=STORE_SIZE, seed=SEED):
    rng = np.random.default_rng(seed)
    base_colors = np.array([[200, 90, 120], [120, 70, 170], [90, 150, 100], [210, 190, 90]], dtype=float)[:n_classes]
    yy, xx = np.mgrid[0:size, 0:size] / size
    Xs, ys, gs = [], [], []
    for g in range(n_groups):
        shift = rng.normal(0, 18, size=3)                       # per-patient "stain" shift
        for _ in range(n_per_group):
            c = rng.integers(0, n_classes)
            freq = 2 + 2 * c                                    # class-dependent texture frequency
            tex = 0.5 + 0.5 * np.sin(2 * np.pi * freq * (xx + rng.random()) + 6 * rng.random())
            img = base_colors[c][None, None, :] * (0.7 + 0.3 * tex[..., None]) + shift
            img += rng.normal(0, 14, size=img.shape)
            Xs.append(np.clip(img, 0, 255).astype(np.uint8)); ys.append(c); gs.append(f"P{g:02d}")
    groups = np.array(gs)
    return LabeledPatches('synthetic', np.stack(Xs), np.array(ys), [f"class_{i}" for i in range(n_classes)],
                          groups, 'patient', grouped_split(groups, 0.25, seed),
                          note='SYNTHETIC smoke test only: not a pathology result.')

ADAPTERS = {'synthetic': build_synthetic}

### Adapter: NCT-CRC-HE (9-class colorectal tissue)

- **Train:** `NCT-CRC-HE-100K` (100,000 patches, 224x224 at 0.5 um/px, 9 tissue classes). **Test:** `CRC-VAL-HE-7K` (7,180 patches). The original data is Zenodo record 1214456 (CC BY 4.0), whose description says the two sets share no patients (not verified here).
- **Where the files come from, in order:** (1) anything already sitting in `<ROOT>/raw/` (`MyDrive/Multi_Dataset_TIL/raw/`), (2) the Hugging Face mirror `DykeF/NCTCRCHE100K` (`.tar.gz` archives, downloaded with `huggingface_hub`, which shows a progress bar), (3) Zenodo zips as a last resort. Archives are **read in place** (streamed, never unpacked); the adapter keeps up to `NCT_PER_CLASS_TRAIN` train patches per class by reservoir sampling, resizes to `STORE_SIZE`, and caches the pool to Drive so later runs skip all of this.
- **`NCT_TRAIN_VARIANT`:** `'NORM'` = the official train set (Macenko stain-normalised); `'NONORM'` = the un-normalised copy that the mirror also hosts. The validation set is un-normalised (per the dataset literature, not checked here), so `NORM` -> test mixes in a **stain shift**, while `NONORM` -> test removes it. Run `NORM` first (official protocol); `NONORM` is the ablation that separates stain shift from model differences.
- **No patient ids:** filenames carry no patient id, so groups are the patches themselves (`group_kind='patch'`): the bootstrap resamples patches and the CIs are **too optimistic**, and the probe's C is chosen by *ungrouped* CV.
- **Contamination:** the adapter counts filenames containing `TCGA` in each split and writes the count into the results note, instead of assuming either way.

In [ ]:
!pip install -q huggingface_hub hf_xet

In [ ]:
import zipfile, tarfile, io, subprocess
from PIL import Image

NCT_CLASSES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
NCT_MEANING = dict(ADI='adipose', BACK='background', DEB='debris', LYM='lymphocytes', MUC='mucus',
                   MUS='smooth muscle', NORM='normal mucosa', STR='cancer-associated stroma',
                   TUM='colorectal adenocarcinoma epithelium')
NCT_TRAIN_VARIANT = 'NORM'          # 'NORM' (official, Macenko-normalised) or 'NONORM' (un-normalised copy)
NCT_PER_CLASS_TRAIN = 3000          # up to 27k train patches pooled; run_experiment then subsamples n_train from this
NCT_HF_REPO = 'DykeF/NCTCRCHE100K'
NCT_ZENODO = 'https://zenodo.org/records/1214456/files/{}?download=1'   # fallback; NORM train + validation only
_IMG_EXT = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')

# Pre-placed archives go in <ROOT>/raw (Drive: MyDrive/Multi_Dataset_TIL/raw); downloads go to local disk.
RAW_DIR = os.path.join(ROOT, 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
NCT_DIR = os.environ.get('NCT_DIR') or ('/content/nct_data' if IN_COLAB else os.path.join(ROOT, 'nct_data'))

_NCT_FILES = {   # kind -> archive names we accept, in preference order (the first is the Hugging Face name)
    'train_NORM':   ['NCT-CRC-HE-100K.tar.gz', 'NCT-CRC-HE-100K.zip'],
    'train_NONORM': ['NCT-CRC-HE-100K-NONORM.tar.gz', 'NCT-CRC-HE-100K-NONORM.zip'],
    'val':          ['CRC-VAL-HE-7K.tar.gz', 'CRC-VAL-HE-7K.zip'],
}

def _valid_archive(path):
    return os.path.exists(path) and (zipfile.is_zipfile(path) or path.endswith(('.tar.gz', '.tgz')))


def _ensure_archive(kind):
    names = _NCT_FILES[kind]
    for d in (RAW_DIR, NCT_DIR):                                   # 1) already on disk / Drive
        for n in names:
            if _valid_archive(os.path.join(d, n)):
                print(f"[{kind}] using existing {os.path.join(d, n)}")
                return os.path.join(d, n)
    os.makedirs(NCT_DIR, exist_ok=True)
    try:                                                           # 2) Hugging Face mirror (progress bar, resumable)
        from huggingface_hub import hf_hub_download
        print(f"[{kind}] downloading {names[0]} from Hugging Face {NCT_HF_REPO} ...")
        return hf_hub_download(NCT_HF_REPO, names[0], repo_type='dataset', local_dir=NCT_DIR)
    except Exception as e:
        print(f"[{kind}] Hugging Face download failed ({type(e).__name__}: {e})")
    if kind == 'train_NONORM':
        raise RuntimeError("NONORM train archive only comes from the Hugging Face mirror; place "
                           f"{names[0]} in {RAW_DIR} manually or set NCT_TRAIN_VARIANT='NORM'.")
    dest = os.path.join(NCT_DIR, names[1])                         # 3) Zenodo zip fallback (progress as dots)
    print(f"[{kind}] falling back to Zenodo -> {dest}")
    subprocess.call(['wget', '-c', '--progress=dot:giga', '-O', dest, NCT_ZENODO.format(names[1])])
    if not zipfile.is_zipfile(dest):
        raise RuntimeError(f"Could not obtain {names[1]}. Download it manually and put it in {RAW_DIR}.")
    return dest


def _decode(data, size):
    with Image.open(io.BytesIO(data)) as im:
        return np.asarray(im.convert('RGB').resize((size, size), Image.LANCZOS))


def _class_of(member_name):
    parts = member_name.split('/')
    if '__MACOSX' in member_name or len(parts) < 2 or parts[-2] not in NCT_CLASSES             or not parts[-1].lower().endswith(_IMG_EXT):
        return None
    return parts[-2]


def _load_archive(path, size, per_class, rng):
    """Read one archive (zip or tar.gz) without unpacking. per_class=None keeps everything,
    otherwise keeps a uniform random sample of up to per_class images per class."""
    slots = {c: [] for c in NCT_CLASSES}
    names = {c: [] for c in NCT_CLASSES}
    seen = {c: 0 for c in NCT_CLASSES}

    def offer(c, name, read):
        seen[c] += 1
        if per_class is None or len(slots[c]) < per_class:
            slots[c].append(_decode(read(), size)); names[c].append(os.path.basename(name))
        else:
            j = int(rng.integers(0, seen[c]))            # reservoir sampling: uniform without knowing the count
            if j < per_class:
                slots[c][j] = _decode(read(), size); names[c][j] = os.path.basename(name)

    if path.endswith(('.tar.gz', '.tgz')):
        n_members = 0
        with tarfile.open(path, 'r|gz') as tf:               # streamed: one sequential pass, no extraction
            for m in tf:
                c = _class_of(m.name) if m.isfile() else None
                if c is not None:
                    offer(c, m.name, lambda m=m: tf.extractfile(m).read())
                n_members += 1
                if n_members % 20000 == 0:
                    print(f"  ... {n_members} archive entries scanned")
    else:
        with zipfile.ZipFile(path) as zf:
            for info in zf.infolist():
                c = None if info.is_dir() else _class_of(info.filename)
                if c is not None:
                    offer(c, info.filename, lambda i=info: zf.read(i))
    print(f"  {os.path.basename(path)}: images per class seen {seen}")
    print(f"  kept per class {({c: len(v) for c, v in slots.items()})}")
    missing = [c for c in NCT_CLASSES if not slots[c]]
    assert not missing, f"no images found for classes {missing}; unexpected archive layout in {path}"
    X = np.concatenate([np.stack(slots[c]) for c in NCT_CLASSES])
    y = np.concatenate([np.full(len(slots[c]), i) for i, c in enumerate(NCT_CLASSES)])
    return X, y, np.array([n for c in NCT_CLASSES for n in names[c]])


def build_nct_crc_he(per_class_train=NCT_PER_CLASS_TRAIN, train_path=None, val_path=None,
                     variant=None, seed=SEED, size=STORE_SIZE):
    variant = variant or NCT_TRAIN_VARIANT
    assert variant in ('NORM', 'NONORM')
    cache = os.path.join(ROOT, 'cache', f"nct_crc_he_{variant}_pc{per_class_train}_z{size}_s{seed}.npz")
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    if os.path.exists(cache):
        print("Loading cached NCT pool:", cache)
        d = np.load(cache)
        X, y, names, is_test = d['X'], d['y'], d['names'], d['is_test']
    else:
        rng = np.random.default_rng(seed)
        train_path = train_path or _ensure_archive(f'train_{variant}')
        val_path = val_path or _ensure_archive('val')
        Xt, yt, nt = _load_archive(train_path, size, per_class_train, rng); print("  loaded train split")
        Xv, yv, nv = _load_archive(val_path, size, None, rng);              print("  loaded validation split")
        X, y, names = np.concatenate([Xt, Xv]), np.concatenate([yt, yv]), np.concatenate([nt, nv])
        is_test = np.concatenate([np.zeros(len(yt), bool), np.ones(len(yv), bool)])
        np.savez_compressed(cache, X=X, y=y, names=names, is_test=is_test)
        print("Cached NCT pool to", cache)

    for label, mask in [('train', ~is_test), ('val', is_test)]:
        print(f"  sample {label} filenames: {[str(n) for n in names[mask][:4]]}")
    n_tcga = {lab: int((np.char.find(names[m], 'TCGA') >= 0).sum()) for lab, m in [('train', ~is_test), ('val', is_test)]}
    stain = ("train=Macenko-normalised, test=un-normalised -> stain-shift confound" if variant == 'NORM'
             else "train and test both un-normalised -> no stain-shift confound")
    note = (f"Official split: train=NCT-CRC-HE-100K ({variant}), test=CRC-VAL-HE-7K; {stain} "
            f"(normalisation status per dataset literature, not checked). "
            f"No patient ids: CIs resample patches (too optimistic) and probe CV is not grouped. "
            f"Filenames containing 'TCGA': train {n_tcga['train']}, val {n_tcga['val']}.")
    return LabeledPatches(f'nct_crc_he_{variant.lower()}', X, y, [f"{c} ({NCT_MEANING[c]})" for c in NCT_CLASSES],
                          names, 'patch', is_test, note=note)

ADAPTERS['nct_crc_he'] = build_nct_crc_he

## 7. Run and summarise

`DATASETS_TO_RUN` lists the adapters to execute. `summarize()` reads every `results_*.json` under `ROOT` and builds (a) one row per dataset x model and (b) the headline table: the probe's gain over the CNN and over the ResNet, per dataset, with CIs, so you can see whether the TCGA-LUAD pattern holds elsewhere.

In [ ]:
def load_results():
    out = {}
    d = os.path.join(ROOT, 'results')
    for fn in sorted(os.listdir(d)):
        if fn.startswith('results_') and fn.endswith('.json'):
            with open(os.path.join(d, fn)) as f:
                r = json.load(f)
            out[r['dataset']] = r
    return out


def summarize(only=None):
    results = {k: v for k, v in load_results().items() if only is None or k in only}
    rows, gains = [], []
    for name, r in results.items():
        for m in MODELS:
            row = dict(dataset=name, classes=r['n_classes'], model=m, baseline=round(r['majority_baseline'], 3))
            row.update({k: round(r['metrics'][m][k]['point'], 3) for k in METRICS})
            rows.append(row)
        for pair in ('Probe-CNN', 'Probe-ResNet'):
            for k in ('accuracy', 'auroc'):
                p = r['paired'][pair][k]
                gains.append(dict(dataset=name, classes=r['n_classes'], contrast=pair, metric=k,
                                  gain=round(p['mean'], 3), ci=f"[{p['lo']:+.3f}, {p['hi']:+.3f}]",
                                  clear=p['excludes_zero'], ci_over=f"{r['group_kind']}s",
                                  caveat=('SYNTHETIC' if name == 'synthetic' else r['note'][:60])))
    return pd.DataFrame(rows), pd.DataFrame(gains)

In [ ]:
DATASETS_TO_RUN = ['synthetic', 'nct_crc_he']   # synthetic = quick sanity check of the code path; then the real run

all_results = {}
for name in DATASETS_TO_RUN:
    print("\n" + "#" * 78 + f"\n# {name}\n" + "#" * 78)
    all_results[name] = run_experiment(ADAPTERS[name](), cfg_for(name))

table, gains = summarize(only=[r['dataset'] for r in all_results.values()])   # result names can differ from adapter keys
print("\n=== All models, all datasets (point estimates) ===")
print(table.to_string(index=False))
print("\n=== Does the probe's gain hold? (probe minus baseline model; 'clear' = 95% CI excludes 0) ===")
print(gains.to_string(index=False))